# FraudShield: Updated Best Practices and Security Improvements

This notebook demonstrates the security and data quality improvements implemented in FraudShield.

## Table of Contents
1. Secure Database Connections
2. Data Leakage Prevention in Feature Engineering
3. Proper Model Evaluation
4. Time-Based Data Splitting
5. Label Encoding Best Practices

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
# Configure Premium Seaborn Aesthetics
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 8)


## 1. Secure Database Connections

### ❌ INSECURE (Old Method)
```python
# DON'T DO THIS - Vulnerable to SQL injection
connection_string = f'postgresql://{user}:{password}@{host}:{port}/{database}'
engine = create_engine(connection_string)
```

###  SECURE (New Method)

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL

# Use environment variables for credentials
db_config = {
    'user': os.getenv('DB_USER', 'default_user'),
    'password': os.getenv('DB_PASSWORD', 'default_password'),
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': os.getenv('DB_PORT', '5432'),
    'database': os.getenv('DB_NAME', 'fraudshield')
}

# Secure connection string construction
db_url = URL.create(
    drivername='postgresql',
    username=db_config['user'],
    password=db_config['password'],
    host=db_config['host'],
    port=db_config['port'],
    database=db_config['database']
)

# Create engine with secure URL
# engine = create_engine(db_url)
print(" Secure database connection configured")

## 2. Data Leakage Prevention in Feature Engineering

### ❌ INCORRECT (Old Method - Data Leakage)
```python
# DON'T DO THIS - Includes current transaction in calculation
def compute_zscore_with_leakage(df, user_col, amount_col):
    grouped = df.groupby(user_col)[amount_col]
    mean = grouped.expanding().mean()  # Includes current row!
    std = grouped.expanding().std(ddof=0)  # Wrong: population std
    z = (df[amount_col] - mean) / std
    return z.replace([np.inf, -np.inf], np.nan)
```

###  CORRECT (New Method - No Data Leakage)

In [ ]:
def compute_zscore_no_leakage(df, user_col, amount_col):
    """
    Compute z-score without data leakage.
    
    Key improvements:
    1. Uses shift(1) to exclude current transaction
    2. Uses sample std (ddof=1) instead of population std
    3. Explicit division by zero handling
    """
    grouped = df.groupby(user_col)[amount_col]
    
    # Exclude current row using shift(1)
    mean = grouped.expanding().mean().groupby(level=0).shift(1)
    std = grouped.expanding().std(ddof=1).groupby(level=0).shift(1)
    
    mean = mean.reset_index(level=0, drop=True)
    std = std.reset_index(level=0, drop=True)
    
    # Handle division by zero explicitly
    z = pd.Series(index=df.index, dtype=float)
    valid_mask = (std > 0) & std.notna()
    z[valid_mask] = (df.loc[valid_mask, amount_col] - mean[valid_mask]) / std[valid_mask]
    z[~valid_mask] = np.nan
    
    return z

# Example usage
sample_data = pd.DataFrame({
    'user_id': [1, 1, 1, 2, 2, 2],
    'amount': [100, 150, 200, 50, 75, 100],
    'transaction_date': pd.date_range('2024-01-01', periods=6)
})

sample_data['amount_zscore'] = compute_zscore_no_leakage(sample_data, 'user_id', 'amount')
print("\n Z-scores computed without data leakage:")
print(sample_data[['user_id', 'amount', 'amount_zscore']])

## 3. Proper Model Evaluation (Avoid Duplicate Predictions)

### ❌ INEFFICIENT (Old Method)
```python
# DON'T DO THIS - Calls predict() multiple times
accuracy = accuracy_score(y_test, model.predict(X_test))
precision = precision_score(y_test, model.predict(X_test))
recall = recall_score(y_test, model.predict(X_test))
```

###  EFFICIENT (New Method)

In [ ]:
# Generate sample data for demonstration
np.random.seed(42)
X_train = np.random.randn(1000, 10)
y_train = np.random.randint(0, 2, 1000)
X_test = np.random.randn(200, 10)
y_test = np.random.randint(0, 2, 200)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample')
rf_model.fit(X_train, y_train)

#  Cache predictions - call predict() only once
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# Reuse cached predictions for all metrics
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob)
}

print("\n Model evaluation metrics (predictions cached):")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

## 4. Time-Based Data Splitting

When working with temporal features, always use time-based splitting to prevent temporal leakage.

In [ ]:
def time_based_split(df, time_column, test_size=0.2):
    """
    Perform time-based train-test split.
    
    Training data always precedes test data chronologically.
    This prevents temporal leakage.
    """
    # Sort by time
    df_sorted = df.sort_values(time_column)
    
    # Calculate split index
    split_idx = max(1, int(len(df_sorted) * (1 - test_size)))
    
    # Split data
    train_df = df_sorted.iloc[:split_idx]
    test_df = df_sorted.iloc[split_idx:]
    
    return train_df, test_df

# Example usage
temporal_data = pd.DataFrame({
    'transaction_date': pd.date_range('2024-01-01', periods=100),
    'amount': np.random.randn(100) * 100 + 500,
    'fraud': np.random.randint(0, 2, 100)
})

train_df, test_df = time_based_split(temporal_data, 'transaction_date', test_size=0.2)

print("\n Time-based split:")
print(f"Training data: {train_df['transaction_date'].min()} to {train_df['transaction_date'].max()}")
print(f"Test data: {test_df['transaction_date'].min()} to {test_df['transaction_date'].max()}")
print(f"\nNo temporal overlap: {train_df['transaction_date'].max() < test_df['transaction_date'].min()}")

## 5. Label Encoding Best Practices

Ensure consistent label encoding between training and test data.

In [ ]:
def safe_label_encoding(y_train, y_test=None):
    """
    Safely encode labels with validation.
    
    Returns:
        y_train_encoded, y_test_encoded (if provided), label_encoder
    """
    label_encoder = None
    
    # Check if encoding is needed
    if y_train.dtype.kind in {'O', 'U', 'S'}:  # String types
        label_encoder = LabelEncoder()
        y_train_encoded = label_encoder.fit_transform(y_train)
        print(f" Label encoder created. Classes: {label_encoder.classes_}")
    elif y_train.dtype.kind in {'f'}:  # Float type
        # Convert to int if only 0.0 and 1.0
        unique_values = set(np.unique(y_train))
        if unique_values.issubset({0.0, 1.0}):
            y_train_encoded = y_train.astype(int)
            print(" Converted float labels to int (0, 1)")
        else:
            y_train_encoded = y_train
    else:
        y_train_encoded = y_train
    
    # Handle test data if provided
    if y_test is not None:
        if label_encoder is not None:
            y_test_encoded = label_encoder.transform(y_test)
            print(" Test labels encoded using training encoder")
        elif y_test.dtype.kind in {'O', 'U', 'S'}:
            raise ValueError("Test data has string labels but no encoder was created from training data")
        elif y_test.dtype.kind in {'f'}:
            unique_values = set(np.unique(y_test))
            if unique_values.issubset({0.0, 1.0}):
                y_test_encoded = y_test.astype(int)
            else:
                y_test_encoded = y_test
        else:
            y_test_encoded = y_test
        
        return y_train_encoded, y_test_encoded, label_encoder
    
    return y_train_encoded, label_encoder

# Example with string labels
y_train_str = np.array(['fraud', 'legitimate', 'fraud', 'legitimate', 'fraud'])
y_test_str = np.array(['legitimate', 'fraud', 'legitimate'])

y_train_enc, y_test_enc, encoder = safe_label_encoding(y_train_str, y_test_str)
print(f"\nTrain labels: {y_train_enc}")
print(f"Test labels: {y_test_enc}")

# Example with float labels
y_train_float = np.array([0.0, 1.0, 0.0, 1.0, 1.0])
y_test_float = np.array([1.0, 0.0, 1.0])

print("\n" + "="*50)
y_train_enc2, y_test_enc2, encoder2 = safe_label_encoding(y_train_float, y_test_float)
print(f"\nTrain labels: {y_train_enc2}")
print(f"Test labels: {y_test_enc2}")

In [ ]:
# Feature Importance Graphical View
from sklearn.metrics import confusion_matrix

def display_model_insights(model, X_test, y_test, feature_cols):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # 1. Confusion Matrix Plotting
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd', ax=ax1,
                xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
    ax1.set_title('Pipeline Test Set: Confusion Matrix', pad=15)
    ax1.set_ylabel('Actual')
    ax1.set_xlabel('Predicted')
    
    # 2. Feature Importances mapped out beautifully
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    sns.barplot(x=importances[indices], y=np.array(feature_cols)[indices], 
                ax=ax2, hue=np.array(feature_cols)[indices], palette='viridis', legend=False)
    ax2.set_title('Pipeline Output: Model Feature Importances', pad=15)
    ax2.set_xlabel('Relative Importance')
    
    plt.tight_layout()
    plt.show()

## 6. Complete Pipeline Example

Putting it all together with best practices.

In [ ]:
def fraud_detection_pipeline_best_practices():
    """
    Complete fraud detection pipeline with all best practices.
    """
    print("Starting FraudShield Pipeline with Best Practices\n")
    print("="*60)
    
    # 1. Generate sample data
    print("\n1. Generating sample transaction data...")
    n_samples = 1000
    data = pd.DataFrame({
        'transaction_date': pd.date_range('2024-01-01', periods=n_samples, freq='H'),
        'user_id': np.random.randint(1, 100, n_samples),
        'amount': np.random.exponential(100, n_samples),
        'fraud': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
    })
    print(f"    Generated {n_samples} transactions")
    
    # 2. Feature engineering without data leakage
    print("\n2. Engineering features (no data leakage)...")
    data = data.sort_values('transaction_date')
    data['amount_zscore'] = compute_zscore_no_leakage(data, 'user_id', 'amount')
    print("    Z-scores computed with shift(1)")
    
    # 3. Time-based split
    print("\n3. Performing time-based train-test split...")
    train_df, test_df = time_based_split(data, 'transaction_date', test_size=0.2)
    print(f"    Train: {len(train_df)} samples, Test: {len(test_df)} samples")
    
    # 4. Prepare features and labels
    print("\n4. Preparing features and labels...")
    feature_cols = ['amount', 'amount_zscore']
    X_train = train_df[feature_cols].fillna(0).values
    X_test = test_df[feature_cols].fillna(0).values
    y_train = train_df['fraud'].values
    y_test = test_df['fraud'].values
    print(f"    Features: {feature_cols}")
    
    # 5. Train model with proper class balancing
    print("\n5. Training Random Forest model...")
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced_subsample',
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    print("    Model trained with balanced class weights")
    
    # 6. Evaluate with cached predictions
    print("\n6. Evaluating model (predictions cached)...")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_test, y_prob)
    }
    
    print("\n   Evaluation Metrics:")
    for metric, value in metrics.items():
        print(f"   {metric:12s}: {value:.4f}")
    
    print("\n" + "="*60)
    print(" Pipeline completed successfully with all best practices!")
    print("="*60)
    
    display_model_insights(model, X_test, y_test, feature_cols)
    return model, metrics

# Run the complete pipeline
